# Khabar Segmentation - Fine-tuning AraBERT on Google Colab

Pipeline complet pour entraîner un modèle AraBERT sur la segmentation d'akhbars arabes.

**GPU recommandé :** T4 ou L4 (gratuit)

**Durée estimée :** ~10-15 minutes (vs 20 min en local)

## 1. Setup & Montage Google Drive

In [ ]:
# Monter Google Drive pour accéder aux données
from google.colab import drive
drive.mount('/content/drive')
print("[OK] Google Drive montée!")

## 2. Vérifier le GPU

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Cloner le repo Khabar-segmentation

In [ ]:
import os
import subprocess

# Cloner le repo (remplacer par ton URL GitHub si public)
repo_path = '/content/Khabar-segmentation'

if not os.path.exists(repo_path):
    print("[*] Cloning repository...")
    # Option 1 : Si le repo est public sur GitHub
    # subprocess.run(['git', 'clone', 'https://github.com/ton-user/Khabar-segmentation.git', repo_path])
    
    # Option 2 : Copier depuis Google Drive (RECOMMANDÉ)
    drive_path = '/content/drive/MyDrive/Khabar-segmentation'
    if os.path.exists(drive_path):
        subprocess.run(['cp', '-r', drive_path, repo_path])
        print(f"[OK] Repo copied from Drive: {repo_path}")
    else:
        print("[!] Repo not found in Drive. Upload it or use GitHub URL.")
        print(f"    Expected path: {drive_path}")
else:
    print(f"[OK] Repo already exists: {repo_path}")

os.chdir(repo_path)
print(f"[OK] Working directory: {os.getcwd()}")

## 4. Installer les dépendances

In [ ]:
!pip install -q torch transformers datasets accelerate scikit-learn pyarabic tqdm hydra-core omegaconf tensorboard

## 5. Préparer le dataset (sans découpage)

In [ ]:
import sys
sys.path.insert(0, '/content/Khabar-segmentation')

# Exécuter le script de préparation
exec(open('scripts/prepare_dataset_v2.py').read())

## 6. Fine-tuner AraBERT

In [ ]:
# Exécuter l'entraînement
exec(open('scripts/train_v2.py').read())

## 7. Quick Test - Extraire quelques akhbars

In [ ]:
# Tester le modèle sur quelques phrases
exec(open('scripts/quick_test.py').read())

## 8. Évaluer sur le corpus OpenITI brut

In [ ]:
# Exécuter l'évaluation
# Note : Assure-toi que la version brute du Kitab Uqala est accessible
exec(open('scripts/evaluate.py').read())

## 9. Sauvegarder les résultats sur Google Drive

In [ ]:
import shutil
from pathlib import Path

# Sauvegarder le checkpoint du modèle
checkpoint_src = Path('checkpoints/arabertv2_akhbars_v2')
checkpoint_dst = Path('/content/drive/MyDrive/Khabar-segmentation-results/checkpoints')

if checkpoint_src.exists():
    checkpoint_dst.parent.mkdir(parents=True, exist_ok=True)
    if checkpoint_dst.exists():
        shutil.rmtree(checkpoint_dst)
    shutil.copytree(checkpoint_src, checkpoint_dst)
    print(f"[OK] Checkpoint saved to: {checkpoint_dst}")

# Sauvegarder les résultats d'évaluation
results_src = Path('results')
results_dst = Path('/content/drive/MyDrive/Khabar-segmentation-results/results')

if results_src.exists():
    results_dst.parent.mkdir(parents=True, exist_ok=True)
    if results_dst.exists():
        shutil.rmtree(results_dst)
    shutil.copytree(results_src, results_dst)
    print(f"[OK] Results saved to: {results_dst}")

print("\n[OK] All results backed up to Google Drive!")

## 10. Résumé & Next Steps

In [ ]:
print("""
    ╔═══════════════════════════════════════════════════════════════╗
    ║           KHABAR SEGMENTATION - PIPELINE COMPLETE              ║
    ╚═══════════════════════════════════════════════════════════════╝

    [OK] Fine-tuning complete!
    [OK] Model: checkpoints/arabertv2_akhbars_v2
    [OK] Results backed up to Google Drive

    NEXT STEPS:
    1. Télécharger le checkpoint depuis Drive
    2. Tester sur d'autres textes arabes
    3. Affiner les hyperparamètres si nécessaire
    4. Déployer le modèle en production

    Questions? Check la documentation dans CLAUDE.md
    """)